In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from alpha_signal.cache.sqlite import SQLiteArticleCache

sns.set_theme(style="whitegrid", palette="deep")

In [ ]:
DB_PATH = "../../articles.db"
cache = SQLiteArticleCache(DB_PATH)
print(f"Cache contains {cache.count()} articles")

In [ ]:
# Entire cache → list[Article]
all_articles = cache.all()
print(f"Loaded {len(all_articles)} Article objects")
all_articles[0]

In [ ]:
# Entire cache → clean DataFrame (authors/categories decoded, raw decompressed)
df = cache.to_dataframe()
df.head()

In [ ]:
# Single row → Article
single = SQLiteArticleCache.dataframe_to_articles(df.iloc[[0]])
single[0]

In [ ]:
extractions = cache.extractions_to_dataframe()

In [ ]:
extractions.iloc[0]['technologies']

In [ ]:
# Explode the technologies list so each mention gets its own row
tech_rows = []
for _, row in extractions.iterrows():
    for tech in (row["technologies"] or []):
        tech_rows.append({
            "source": row["source"],
            "source_id": row["source_id"],
            "title": row["title"],
            "technology": tech["technology"],
            "sector": tech["sector"],
            "maturity": tech["maturity"],
            "relevance": tech["relevance"],
        })

tech_df = pd.DataFrame(tech_rows)
print(f"{len(tech_df)} technology mentions across {extractions.shape[0]} extractions")
tech_df.head()

In [ ]:
# Sector distribution — horizontal bar chart sorted by count
sector_counts = tech_df["sector"].value_counts().reset_index()
sector_counts.columns = ["sector", "count"]

fig, ax = plt.subplots(figsize=(10, max(6, len(sector_counts) * 0.35)))
sns.barplot(
    data=sector_counts.sort_values("count"),
    y="sector", x="count", ax=ax, orient="h",
)
ax.set_xlabel("Number of technology mentions")
ax.set_ylabel("")
ax.set_title("Technology Mentions by Sector")
ax.bar_label(ax.containers[0], padding=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Maturity breakdown within each sector (top 15 sectors)
top_sectors = sector_counts.nlargest(15, "count")["sector"]
top_tech = tech_df[tech_df["sector"].isin(top_sectors)].copy()

maturity_order = ["theoretical", "lab_scale", "pilot", "commercial"]
top_tech["maturity"] = pd.Categorical(top_tech["maturity"], categories=maturity_order, ordered=True)

sector_order = (
    top_tech.groupby("sector").size()
    .sort_values(ascending=True).index.tolist()
)

fig, ax = plt.subplots(figsize=(10, max(6, len(sector_order) * 0.45)))
sns.histplot(
    data=top_tech, y="sector", hue="maturity",
    multiple="stack", shrink=0.8, ax=ax,
    hue_order=maturity_order,
    palette=["#aec7e8", "#1f77b4", "#ff7f0e", "#2ca02c"],
)
ax.set_yticklabels(ax.get_yticklabels())
ax.set_order = sector_order
ax.set_xlabel("Number of technology mentions")
ax.set_ylabel("")
ax.set_title("Sector × Maturity Distribution (top 15 sectors)")
sns.move_legend(ax, "lower right", title="Maturity")
plt.tight_layout()
plt.show()

In [ ]:
mat_sci = tech_df[tech_df['sector'] == 'Materials Science']

In [ ]:
mat_sci[mat_sci['maturity'] == 'commercial']

In [ ]:
extractions[extractions['source_id'] == "2510.27242v1"]